# บทเรียน 01 - แนะนำตัวแทน AI

ยินดีต้อนรับสู่บทเรียนแรกในหลักสูตร **AI Agents for Beginners**

**AI agent** คือโปรแกรมที่ใช้โมเดลภาษาใหญ่ (LLM) เป็นเครื่องยนต์เหตุผลและสามารถดำเนิน *การกระทำ* ในโลกจริง — การเรียกใช้ API, การสอบถามฐานข้อมูล หรือการรันโค้ด — เพื่อบรรลุเป้าหมายแทนผู้ใช้

ในโน้ตบุ๊กนี้คุณจะสร้างตัวแทนตัวแรก: **Travel Agent** ที่แนะนำจุดหมายปลายทางสำหรับวันหยุด ระหว่างทางคุณจะได้เรียนรู้วิธี:

1. เชื่อมต่อกับ Azure AI Foundry Agent Service โดยใช้ **Microsoft Agent Framework**
2. ให้ตัวแทนมี **เครื่องมือ** — ฟังก์ชัน Python ง่ายๆ ที่ตัวแทนสามารถเรียกใช้ได้
3. รันตัวแทนและตรวจสอบการตอบกลับ
4. สตรีมการตอบกลับของตัวแทนทีละโทเค็น


## การตั้งค่า

ก่อนที่จะรันโน้ตบุ๊กนี้ ให้แน่ใจว่าคุณมี:

1. **โปรเจกต์ Azure AI Foundry** ที่มีโมเดลแชทปรับใช้งานแล้ว (เช่น `gpt-4o-mini`)
2. **เข้าสู่ระบบด้วย Azure CLI** — รันคำสั่ง `az login` ในเทอร์มินัลของคุณ
3. **ตั้งค่าตัวแปรแวดล้อมที่จำเป็น:**
   - `AZURE_AI_PROJECT_ENDPOINT` — จุดสิ้นสุดโปรเจกต์ Azure AI Foundry ของคุณ
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ชื่อของโมเดลที่คุณปรับใช้งาน

เซลล์ด้านล่างจะติดตั้งแพ็กเกจ Python ที่คุณต้องการใช้งาน


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## การสร้างเอเย่นต์ตัวแรกของคุณ

เอเย่นต์ต้องการสองสิ่ง:

- **คำสั่ง** ที่บอกว่ามัน *คือใคร* และ *ควรประพฤติอย่างไร* (พรอมต์ระบบ)
- **เครื่องมือ** — ฟังก์ชัน Python ที่ตกแต่งด้วย `@tool` ซึ่งเอเย่นต์สามารถเรียกใช้เพื่อดึงข้อมูลหรือดำเนินการต่างๆ

ด้านล่างนี้เรากำหนดเครื่องมืออย่างง่ายที่ส่งคืนรายชื่อสถานที่ท่องเที่ยวยอดนิยม เอเย่นต์จะใช้เครื่องมือนี้เมื่อผู้ใช้ขอคำแนะนำเกี่ยวกับการเดินทาง


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## การตอบสนองแบบสตรีมมิ่ง

สำหรับประสบการณ์ที่มีปฏิสัมพันธ์มากขึ้น คุณสามารถ **สตรีม** การตอบกลับของเอเย่นต์ แทนที่จะรอคำตอบเต็ม เอเย่นต์จะส่งข้อความเป็นชิ้นส่วนตามที่ถูกสร้างขึ้น ซึ่งจะมีประโยชน์อย่างยิ่งในอินเทอร์เฟซแชทที่คุณต้องการแสดงผลแบบเรียลไทม์


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้วิธี:

- **สร้าง provider** ที่เชื่อมต่อกับ Azure AI Foundry Agent Service ผ่าน `AzureAIProjectAgentProvider`
- **กำหนดเครื่องมือ** โดยใช้ตัวตกแต่ง `@tool` เพื่อให้เอเจนต์เรียกใช้ฟังก์ชัน Python ของคุณได้
- **เรียกใช้งานเอเจนต์** พร้อมกับข้อความจากผู้ใช้และพิมพ์การตอบกลับของมัน
- **สตรีมคำตอบ** เพื่อแสดงผลแบบเรียลไทม์

ในบทเรียนถัดไปเราจะสำรวจโครงสร้างเอเจนต์ในเชิงลึกมากขึ้นและเรียนรู้วิธีให้เอเจนต์มีเครื่องมือที่ทรงพลังมากขึ้นและความสามารถในการให้เหตุผลแบบหลายขั้นตอนมากขึ้น


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**คำปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษาด้วยปัญญาประดิษฐ์ [Co-op Translator](https://github.com/Azure/co-op-translator) แม้ว่าเราจะพยายามให้ถูกต้อง แต่โปรดทราบว่าการแปลอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางถือเป็นแหล่งข้อมูลที่น่าเชื่อถือที่สุด สำหรับข้อมูลที่สำคัญ ขอแนะนำให้ใช้การแปลโดยนักแปลมืออาชีพ เรายินดีไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความผิดที่อาจเกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
